# Session 3: Text Preprocessing — POS Tagging & Named Entity Recognition



## Setup
Import libraries and download the required NLTK/spaCy data.


In [1]:
import nltk
from nltk import pos_tag, word_tokenize, ne_chunk

import spacy
from spacy import displacy

from collections import Counter
import pandas as pd

# One-time NLTK data downloads (safe to re-run; skips if already present)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('maxent_ne_chunker', quiet=True)
nltk.download('maxent_ne_chunker_tab', quiet=True)
nltk.download('words', quiet=True)

nlp = spacy.load('en_core_web_sm')

print("NLTK and spaCy ready.")


NLTK and spaCy ready.


---
## Section 1: Text Preprocessing — Parts of Speech (POS) Tagging

### Q1. What is POS Tagging? Tagging with NLTK
What is Parts of Speech (POS) tagging, and why is it a useful preprocessing step in an NLP pipeline? Apply `nltk.pos_tag` to the sentence below, print the tagged output, and explain what at least five of the resulting Penn Treebank tags mean.


In [ ]:
sentence = "The quick brown fox jumps gracefully over the lazy dog near the river."

tokens = word_tokenize(sentence)
tagged = pos_tag(tokens)

print("POS-tagged sentence:")
for word, tag in tagged:
    print(f"  {word:12s} -> {tag}")


### Q2. POS Tagging with spaCy — Comparison to NLTK
Tag the same sentence from Q1 using spaCy's POS tagger. Print each token alongside its spaCy POS tag. Compare NLTK's Penn Treebank tag set to spaCy's (coarse `pos_` and fine-grained `tag_`) — do the tag sets match, and do the two libraries agree on every word?


In [ ]:
doc = nlp(sentence)

spacy_results = pd.DataFrame({
    'token': [t.text for t in doc],
    'spacy_coarse_pos': [t.pos_ for t in doc],
    'spacy_fine_tag': [t.tag_ for t in doc],
    'nltk_tag': [tag for _, tag in tagged]
})
spacy_results


### Q3. Keyword Extraction Using POS Filtering
Using POS tags, build a simple keyword extractor that pulls out only the nouns and verbs from the paragraph below. Why might filtering to just nouns and verbs be useful for topic extraction or search indexing?


In [ ]:
paragraph = (
    "Researchers at the university recently published a groundbreaking study. "
    "The team analyzed climate data collected over three decades and discovered "
    "significant patterns that could help predict future weather events."
)

p_tokens = word_tokenize(paragraph)
p_tagged = pos_tag(p_tokens)

keywords = [word for word, tag in p_tagged if tag.startswith('NN') or tag.startswith('VB')]

print("Full POS-tagged paragraph:")
print(p_tagged)
print(f"\nExtracted keywords (nouns + verbs), {len(keywords)} of {len(p_tokens)} tokens kept:")
print(keywords)


### Q4. Context-Based Disambiguation: "book" and "run"
POS taggers rely on context to disambiguate words with multiple grammatical roles. Using spaCy, tag the sentences below and show how "book" and "run" are tagged differently depending on their role. Explain how the tagger makes this distinction.


In [ ]:
disambiguation_sentences = [
    "I need to book a flight for my trip.",
    "I am reading a great book about history.",
    "She went for a run this morning.",
    "The engineers will run the experiment tomorrow."
]

for s in disambiguation_sentences:
    d = nlp(s)
    for t in d:
        if t.text.lower() in ('book', 'run'):
            print(f"'{s}'\n  -> '{t.text}' tagged as {t.pos_} ({t.tag_})\n")


---
## Section 2: Text Preprocessing — Named Entity Recognition (NER)

### Q5. What is NER? Extracting Entities with spaCy
What is Named Entity Recognition (NER), and what kinds of real-world information does it extract? List at least five common entity types. Apply spaCy's NER pipeline to the paragraph below and print every entity found along with its label.


In [ ]:
news_paragraph = (
    "Apple Inc. announced on September 12, 2023 that Tim Cook would unveil the "
    "new iPhone in Cupertino, California. The event, expected to generate over "
    "$2 billion in revenue, will be held at Apple Park. Analysts from Goldman "
    "Sachs praised the launch, noting that Microsoft and Google are also "
    "releasing competing products this year."
)

news_doc = nlp(news_paragraph)

print(f"{'Entity':30s} Label")
print("-" * 45)
for ent in news_doc.ents:
    print(f"{ent.text:30s} {ent.label_}")


### Q6. Visualizing Entities & Comparing spaCy to NLTK's `ne_chunk`
Visualize the named entities from Q5 using spaCy's `displacy` renderer. Separately, run NLTK's `ne_chunk` on the same paragraph and print its output. Compare the two approaches.


In [ ]:
# --- spaCy visualization ---
displacy.render(news_doc, style='ent', jupyter=True)


In [ ]:
# --- NLTK's ne_chunk (built on top of POS tags) ---
news_tokens = word_tokenize(news_paragraph)
news_tagged = pos_tag(news_tokens)
nltk_chunks = ne_chunk(news_tagged)

print(nltk_chunks)


### Q7. Entity Frequency Counting
Extract only the `PERSON` and `ORG` entities from the article below with spaCy, and compute how many times each distinct entity is mentioned. Print the entities ranked from most to least frequently mentioned.


In [ ]:
article = (
    "Elon Musk met with officials in Berlin to discuss Tesla's new Gigafactory. "
    "Tesla has invested heavily in the region, and Musk confirmed that Tesla "
    "plans to double production capacity by next year. Meanwhile, BMW and "
    "Volkswagen, two major German automakers, announced a joint venture with "
    "Tesla to develop shared battery technology. A spokesperson for BMW said "
    "the partnership with Tesla marks a new chapter for the German auto industry."
)

article_doc = nlp(article)

persons = [ent.text for ent in article_doc.ents if ent.label_ == 'PERSON']
orgs = [ent.text for ent in article_doc.ents if ent.label_ == 'ORG']

person_counts = Counter(persons)
org_counts = Counter(orgs)

print("PERSON mentions (most to least frequent):")
for name, count in person_counts.most_common():
    print(f"  {name:15s} {count}")

print("\nORG mentions (most to least frequent):")
for name, count in org_counts.most_common():
    print(f"  {name:15s} {count}")


### Q8. Combining POS Tagging and NER: A Simple "Who Did What" Pipeline
For each `PERSON` or `ORG` entity found in the paragraph below, extract the nearest verb in the same sentence to get a rough "who did what" summary. Discuss one limitation of this simple nearest-verb heuristic.


In [ ]:
summary_text = (
    "Sundar Pichai announced a new AI product at Google's annual conference. "
    "Meanwhile, Microsoft launched a competing tool, and Satya Nadella "
    "praised the rapid pace of innovation in the industry."
)

summary_doc = nlp(summary_text)

print("All entities found:", [(ent.text, ent.label_) for ent in summary_doc.ents])
print()

results = []
for sent in summary_doc.sents:
    verbs_in_sentence = [t for t in sent if t.pos_ == 'VERB']
    for ent in sent.ents:
        if ent.label_ in ('PERSON', 'ORG') and verbs_in_sentence:
            # nearest verb by token-position distance
            nearest_verb = min(verbs_in_sentence, key=lambda v: abs(v.i - ent.start))
            results.append({'entity': ent.text, 'label': ent.label_, 'nearest_verb': nearest_verb.text})

who_did_what = pd.DataFrame(results)
who_did_what
